# Visualization 3 - Gender Scatter Plot

This notebook creates a simple visualization for the third mini-project question:

- How do naming patterns differ between boys and girls?
- For names used by both sexes, do popularity trends evolve in the same way?
- Can a name shift from mostly male to mostly female, or the reverse?

The chart uses a few **signature unisex names** and compares male and female births with a connected scatter plot.

Here, **each point is one year**, and the points are linked so you can follow how the name moves over time.

In [1]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()
alt.renderers.enable('default')

RendererRegistry.enable('default')

In [2]:
# Load and clean the dataset.
names = pd.read_csv('dpt2020.csv', sep=';')
names = names[(names['preusuel'] != '_PRENOMS_RARES') & (names['dpt'] != 'XX') & (names['annais'] != 'XXXX')].copy()
names['annais'] = names['annais'].astype(int)
names['nombre'] = names['nombre'].astype(int)

# National yearly totals by name and sex.
gender_year = names.groupby(['annais', 'preusuel', 'sexe'], as_index=False)['nombre'].sum()

# Build a pool of unisex names that have meaningful usage for both sexes.
name_sex_totals = gender_year.groupby(['preusuel', 'sexe'], as_index=False)['nombre'].sum()
name_sex_pivot = name_sex_totals.pivot_table(index='preusuel', columns='sexe', values='nombre', fill_value=0).reset_index()
name_sex_pivot.columns = ['preusuel', 'male_total', 'female_total']
name_sex_pivot = name_sex_pivot[(name_sex_pivot['male_total'] >= 200) & (name_sex_pivot['female_total'] >= 200)].copy()
name_sex_pivot['total'] = name_sex_pivot['male_total'] + name_sex_pivot['female_total']
available_names = name_sex_pivot.sort_values('total', ascending=False)['preusuel'].head(120).tolist()

# A few unisex names with clearly different trajectories.
selected_names = ['DOMINIQUE', 'CLAUDE', 'CHARLIE', 'LOU', 'SACHA']
plot_data = (
    gender_year[gender_year['preusuel'].isin(available_names)]
    .pivot_table(index=['annais', 'preusuel'], columns='sexe', values='nombre', fill_value=0)
    .reset_index()
)
plot_data.columns = ['annais', 'preusuel', 'male_births', 'female_births']
plot_data = plot_data[(plot_data['annais'] >= 1900) & (plot_data['annais'] <= 2020)].copy()
plot_data = plot_data.sort_values(['preusuel', 'annais']).copy()

# Keep only years where the name is meaningfully present for at least one sex.
plot_data = plot_data[(plot_data['male_births'] + plot_data['female_births']) >= 20].copy()
min_year = int(plot_data['annais'].min())
max_year = int(plot_data['annais'].max())

plot_data.head()

,annais,preusuel,male_births,female_births
3620,1981,ADAMA,16.0,4.0
3667,1982,ADAMA,18.0,3.0
3717,1983,ADAMA,29.0,3.0
3766,1984,ADAMA,23.0,3.0
3818,1985,ADAMA,35.0,3.0


In [ ]:
import ipywidgets as widgets
from IPython.display import display

selected_name_labels = selected_names.copy()
all_name_labels = available_names.copy()
start_year = 2000
end_year = 2020
chart_handle = None

palette_pool = ['#d62828', '#1d4ed8', '#16a34a', '#f59e0b', '#7c3aed', '#0891b2', '#db2777', '#65a30d', '#ea580c', '#4f46e5']

def build_chart(selected_names_for_chart, year_start, year_end):
    filtered = plot_data[
        plot_data['preusuel'].isin(selected_names_for_chart)
        & plot_data['annais'].between(year_start, year_end)
    ].copy()

    filtered['year_strength'] = 0.35
    if year_end > year_start:
        filtered['year_strength'] = 0.25 + 0.75 * ((filtered['annais'] - year_start) / (year_end - year_start))

    limits = max(filtered['male_births'].max(), filtered['female_births'].max()) if not filtered.empty else 1
    reference = pd.DataFrame({'x': [0, limits], 'y': [0, limits]})

    diag = alt.Chart(reference).mark_line(color='#999', strokeDash=[4, 4]).encode(
        x=alt.X('x:Q', title='Male births', scale=alt.Scale(domain=[0, limits], nice=False)),
        y=alt.Y('y:Q', title='Female births', scale=alt.Scale(domain=[0, limits], nice=False))
    )

    segments = filtered.copy()
    segments['male_births_next'] = segments.groupby('preusuel')['male_births'].shift(-1)
    segments['female_births_next'] = segments.groupby('preusuel')['female_births'].shift(-1)
    segments = segments.dropna(subset=['male_births_next', 'female_births_next']).copy()

    palette = palette_pool[:len(selected_names_for_chart)]

    lines = alt.Chart(segments).mark_rule(strokeWidth=2.5).encode(
        x=alt.X('male_births:Q', title='Male births', scale=alt.Scale(domain=[0, limits], nice=False)),
        y=alt.Y('female_births:Q', title='Female births', scale=alt.Scale(domain=[0, limits], nice=False)),
        x2='male_births_next:Q',
        y2='female_births_next:Q',
        color=alt.Color('preusuel:N', title='Name', scale=alt.Scale(domain=selected_names_for_chart, range=palette)),
        opacity=alt.Opacity('year_strength:Q', legend=None, scale=alt.Scale(domain=[0.25, 1], range=[0.2, 1]))
    )

    points = alt.Chart(filtered).mark_circle(size=70).encode(
        x=alt.X('male_births:Q', title='Male births', scale=alt.Scale(domain=[0, limits], nice=False)),
        y=alt.Y('female_births:Q', title='Female births', scale=alt.Scale(domain=[0, limits], nice=False)),
        color=alt.Color('preusuel:N', title='Name', scale=alt.Scale(domain=selected_names_for_chart, range=palette)),
        opacity=alt.Opacity('year_strength:Q', legend=None, scale=alt.Scale(domain=[0.25, 1], range=[0.25, 1])),
        tooltip=[
            alt.Tooltip('preusuel:N', title='Name'),
            alt.Tooltip('annais:Q', title='Year'),
            alt.Tooltip('male_births:Q', title='Male births'),
            alt.Tooltip('female_births:Q', title='Female births')
        ]
    )

    return (diag + lines + points).properties(
        width=620,
        height=460,
        title=f'Shared Names Can Move Over Time Between Male and Female Usage ({year_start}-{year_end})'
    )

def refresh_controls():
    add_name_dropdown.options = [label for label in all_name_labels if label not in selected_name_labels]
    remove_name_dropdown.options = selected_name_labels.copy()
    if add_name_dropdown.options:
        add_name_dropdown.value = add_name_dropdown.options[0]
    else:
        add_name_dropdown.value = None
    if remove_name_dropdown.options:
        remove_name_dropdown.value = remove_name_dropdown.options[0]
    else:
        remove_name_dropdown.value = None

def refresh_chart():
    chart_handle.update(build_chart(selected_name_labels, year_start_slider.value, year_end_slider.value))

def on_add_name_clicked(_):
    value = add_name_dropdown.value
    if value and value not in selected_name_labels:
        selected_name_labels.append(value)
        refresh_controls()
        refresh_chart()

def on_remove_name_clicked(_):
    value = remove_name_dropdown.value
    if value and value in selected_name_labels and len(selected_name_labels) > 1:
        selected_name_labels.remove(value)
        refresh_controls()
        refresh_chart()

def on_year_change(change):
    if year_start_slider.value > year_end_slider.value:
        if change['owner'] is year_start_slider:
            year_end_slider.value = year_start_slider.value
        else:
            year_start_slider.value = year_end_slider.value
    refresh_chart()

add_name_dropdown = widgets.Dropdown(
    options=[],
    description='Add name:',
    layout=widgets.Layout(width='320px')
)

remove_name_dropdown = widgets.Dropdown(
    options=[],
    description='Remove name:',
    layout=widgets.Layout(width='320px')
)

year_start_slider = widgets.IntSlider(value=start_year, min=min_year, max=max_year, step=1, description='From year:', layout=widgets.Layout(width='320px'))
year_end_slider = widgets.IntSlider(value=end_year, min=min_year, max=max_year, step=1, description='To year:', layout=widgets.Layout(width='320px'))

add_name_button = widgets.Button(description='Add', button_style='primary')
remove_name_button = widgets.Button(description='Remove')
add_name_button.on_click(on_add_name_clicked)
remove_name_button.on_click(on_remove_name_clicked)
year_start_slider.observe(on_year_change, names='value')
year_end_slider.observe(on_year_change, names='value')

refresh_controls()
display(widgets.VBox([
    widgets.HBox([add_name_dropdown, add_name_button, remove_name_dropdown, remove_name_button]),
    widgets.HBox([year_start_slider, year_end_slider]),
]))
chart_handle = display(build_chart(selected_name_labels, start_year, end_year), display_id=True)


alt.LayerChart(...)

## Why this works for Visualization 3

### Advantages

- The path makes gender shifts over time easy to see for each shared name.
- The diagonal reference line clearly separates male-dominant and female-dominant usage.
- It answers the assignment question directly by showing whether the two sexes evolve consistently or not.
- Each point is one year for a shared name.
- The diagonal line shows the balance point: points above it are more female, points below it are more male.
- The connected path shows how a name moves over time between male and female usage.
- The visualization stays simple by focusing on a few **signature unisex names** rather than every shared name.


### Disadvantages

- The chart is based on a curated subset of shared names, not the full dataset.
- Without interaction, comparing many names at once could become cluttered.
- Some viewers may need a short explanation to understand the meaning of the diagonal line at first glance.
- The lines can overlap and make it hard to distinguish individual names if too many are included or the number of births is very small for some names, leading to a cluttered visualization.
- The color may not very clear to see the evolution of the years